In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
import requests

load_dotenv(override=True)

# API keys
openai_api_key = os.getenv("OPENAI_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")

print(
    f"OpenAI API Key exists and begins {openai_api_key[:8]}"
    if openai_api_key
    else "OpenAI API Key not set"
)
print(
    f"Google API Key exists and begins {google_api_key[:2]}"
    if google_api_key
    else "Google API Key not set (optional)"
)

In [ ]:
# Clients
openai = OpenAI()
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

# Models
gpt_model = "gpt-4.1-mini"
gemini_model = "gemini-3.1-flash-lite-preview"

conversation = [
    {"role": "user", "content": "Blake: I think AI is going to take over the world."},
    {"role": "user", "content": "Charlie: I disagree, AI helps us."},
]

# Call functions
def call_agent(client, model, system_prompt, conversation):
    messages = [{"role": "system", "content": system_prompt}] + conversation
    try:
        response = client.chat.completions.create(model=model, messages=messages)
        content = response.choices[0].message.content
        return content if content else "ERROR: Empty response from model"
    except Exception as e:
        return f"ERROR: {e}"

In [ ]:
# Prompts
alex_prompt = """You are Alex.
You disagree with everything and are sarcastic.
Rules:
Always reply as Alex colon space and then your message
Keep it short up to three sentences
Be sharp witty and mocking
"""

blake_prompt = """You are Blake.
You think AI is dangerous.
Rules:
Always reply as Blake colon space and then your message
Keep it short two to four sentences
Be emotional and skeptical not academic
"""

charlie_prompt = """You are Charlie.
Rules:
Always reply as Charlie colon space and then your message
Maximum two sentences
No complex words
No formatting
No symbols like ellipsis or bold
If unsure say something simple and clear
"""

In [ ]:
# Agents
agents = [
    ("Blake", gemini, gemini_model, blake_prompt),
    ("Charlie", openai, gpt_model, charlie_prompt),
    ("Alex", gemini, gemini_model, alex_prompt),
]

# Conversation loop
for i in range(9):
    name, client, model, prompt = agents[i % 3]
    reply = call_agent(client, model, prompt, conversation)

    if not reply or not reply.strip() or reply.startswith("ERROR"):
        display(Markdown(f"_{name} did not respond this turn._"))
        continue

    # Remove redundant name prefix if present
    if reply.startswith(f"{name}:"):
        reply = reply[len(f"{name}:"):].lstrip()

    display(Markdown(f"**{name}**: {reply}"))
    conversation.append({"role": "assistant", "content": f"{name}: {reply}"})

    # Optional: add a separator for readability
    display(Markdown("---"))